In [1]:
import boto3
import json
from concurrent.futures import ThreadPoolExecutor

### `Lambda`

In [2]:
aws_lambda = boto3.client('lambda')
iam_client = boto3.client('iam')
role = iam_client.get_role(RoleName='LabRole')

with open('lab4.py.zip', 'rb') as f:
    lambda_zip = f.read()

In [3]:
try:
    # If function hasn't yet been created, create it
    response = aws_lambda.create_function(
        FunctionName='lambda_handler',
        Runtime='python3.11',
        Role=role['Role']['Arn'],
        Handler='lambda_function.lambda_handler',
        Code=dict(ZipFile=lambda_zip),
        Timeout=300
    )
except aws_lambda.exceptions.ResourceConflictException:
    # If function already exists, update it based on zip
    # file contents
    response = aws_lambda.update_function_code(
        FunctionName='lambda_handler',
        ZipFile=lambda_zip
        )

lambda_arn = response['FunctionArn']
lambda_arn

'arn:aws:lambda:us-east-1:767397986207:function:lambda_handler'

In [4]:
r = aws_lambda.invoke(FunctionName='lambda_handler',
                      InvocationType='RequestResponse',
                      Payload=json.dumps({'num_points':10000}))
json.loads(r['Payload'].read())

{'statusCode': 200, 'body': {'pi_estimate': 3.1252, 'num_points': 10000}}

In [5]:
# 1. write function to invoke our function for us and pass in data:
def invoke_function(data):
    r = aws_lambda.invoke(FunctionName='lambda_handler',
                      InvocationType='RequestResponse',
                      Payload=json.dumps({'num_points':data}))
    return json.loads(r['Payload'].read())

# 2. Demo that lambda function will scale out if called concurrently on different threads locally
# Do not invoke more than 10 workers at a time or you risk your AWS Academy account being deactivated
with ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(invoke_function, [100000 for _ in range(5)])

output = [result for result in results]
print(output)

[{'statusCode': 200, 'body': {'pi_estimate': 3.12944, 'num_points': 100000}}, {'statusCode': 200, 'body': {'pi_estimate': 3.14196, 'num_points': 100000}}, {'statusCode': 200, 'body': {'pi_estimate': 3.14328, 'num_points': 100000}}, {'statusCode': 200, 'body': {'pi_estimate': 3.14504, 'num_points': 100000}}, {'statusCode': 200, 'body': {'pi_estimate': 3.14644, 'num_points': 100000}}]


In [6]:
# pi
print(sum([i['body']['pi_estimate'] for i in output]) / len(output))

3.141232
